# 4. Complex open-platform queries

Compose continuous and categorical FILTER clauses, combine them under AND / OR groups, and nest groups inside groups. Same `buildClause()` / `buildClauseGroup()` / `buildQuery()` / `runQuery()` API used throughout — here we run each filter tree through `buildQuery()` before executing it.

In [1]:
library(picsure)

picsure loaded. On first call, reticulate will provision an isolated Python environment; this takes a few seconds the first time only.



In [2]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [3]:
open_hpds_session <- picsure::connect(
  platform = picsure::Platform$BDC_OPEN
)

## Restrict the search to two studies

In [4]:
two_studies <- picsure::facets(open_hpds_session)
picsure::addFacet(two_studies, "dataset_id", c("phs000810", "phs000007"))

In [5]:
results <- picsure::searchDictionary(open_hpds_session, "age", facets = two_studies)

## Pick the two AGE variables to use

In [6]:
age_immi_phs000810 <- results[results$display == "AGE_IMMI", ]
age_immi_phs000810

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>,<dbl>,<dbl>,<lgl>,<chr>,<chr>
1,\phs000810\pht004715\phv00526256\AGE_IMMI\,phv00526256,AGE_IMMI,Age of immigration among participants who were not born in US mainland (50 US States plus DC),Continuous,phs000810,NULL,0,73,TRUE,NA,HCHSSOL


In [7]:
age5_phs000007 <- results[results$display == "age5" & results$name == "phv00177938", ]
age5_phs000007

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>,<dbl>,<dbl>,<lgl>,<chr>,<chr>
4,\phs000007\pht003099\phv00177938\age5\,phv00177938,age5,Age at Exam 5,Continuous,phs000007,NULL,26,96,TRUE,NA,FHS


## Continuous FILTER (numeric range)

Pass `min` and `max` instead of `categories`.

In [8]:
age5_phs000007_clause <- picsure::buildClause(
  age5_phs000007$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = age5_phs000007_clause))

CountResult(value=615, margin=3, cap=None, raw='615 ±3')

In [9]:
age_immi_phs000810_clause <- picsure::buildClause(
  age_immi_phs000810$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = age_immi_phs000810_clause))

CountResult(value=2255, margin=3, cap=None, raw='2255 ±3')

## Combine with OR

Counts participants who match *either* clause.

In [10]:
clause_group_or <- picsure::buildClauseGroup(
  list(age_immi_phs000810_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$OR
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = clause_group_or))

CountResult(value=2868, margin=3, cap=None, raw='2868 ±3')

## Add a categorical FILTER on sex

In [11]:
fhs_facet <- picsure::facets(open_hpds_session)
picsure::addFacet(fhs_facet, "dataset_id", "phs000007")

fhs_sex_results <- picsure::searchDictionary(open_hpds_session, "phv00253990", facets = fhs_facet)
fhs_sex_results

conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>,<chr>,<chr>,<lgl>,<chr>,<chr>
\phs000007\pht004374\phv00253990\sex\,phv00253990,sex,Sex of the participant,Categorical,phs000007,"Female, Male",NA,NA,TRUE,NA,FHS


In [12]:
fhs_sex_male_clause <- picsure::buildClause(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Male")
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_sex_male_clause))

CountResult(value=576, margin=3, cap=None, raw='576 ±3')

## Combine with AND

Males aged 30–40.

In [13]:
fhs_male_and_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_sex_male_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_male_and_30_to_40))

CountResult(value=28, margin=3, cap=None, raw='28 ±3')

In [14]:
fhs_sex_female_clause <- picsure::buildClause(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Female")
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_sex_female_clause))

CountResult(value=693, margin=3, cap=None, raw='693 ±3')

In [15]:
fhs_female_and_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_sex_female_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_female_and_30_to_40))

CountResult(value=53, margin=3, cap=None, raw='53 ±3')

## Nested groups

`buildClauseGroup()` accepts both leaf clauses *and* other groups. Nest groups inside groups to express arbitrary Boolean trees — here, OR-of-ANDs.

In [16]:
fhs_female_ages_30_to_40_or_male_ages_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_female_and_30_to_40, fhs_male_and_30_to_40),
  operator = picsure::GroupOperator$OR
)

picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_female_ages_30_to_40_or_male_ages_30_to_40))

CountResult(value=78, margin=3, cap=None, raw='78 ±3')